In [3]:
import pandas as pd
import duckdb

# 设备每日运行日志表
log_data = [
    ["A", "2026-07-01", "NORMAL", 2],
    ["A", "2026-07-02", "ERROR", 5],
    ["A", "2026-07-03", "ERROR", 6],

    ["B", "2026-07-01", "NORMAL", 1],
    ["B", "2026-07-02", "NORMAL", 2],
    ["B", "2026-07-03", "ERROR", 4],

    ["C", "2026-07-01", "ERROR", 7],
    ["C", "2026-07-02", "NORMAL", 3],

    # E 在日志表中存在，但设备信息表中没有
    ["E", "2026-07-01", "ERROR", 9],
    ["E", "2026-07-02", "ERROR", 8],
]

df_log = pd.DataFrame(
    log_data,
    columns=["device_id", "stat_date", "status", "alarm_count"]
)

df_log["stat_date"] = pd.to_datetime(df_log["stat_date"])


# 设备基础信息表
device_data = [
    ["A", "R34", "VIS", "FS11"],
    ["B", "R34", "RVR", "LT31"],
    ["C", "R35", "VIS", "FS11"],

    # D 在设备信息表中存在，但日志表中没有
    ["D", "R35", "RVR", "LT31"],
]

df_device = pd.DataFrame(
    device_data,
    columns=["device_id", "site", "device_type", "model"]
)

print("df_log:")
print(df_log)

print("\ndf_device:")
print(df_device)

df_log:
  device_id  stat_date  status  alarm_count
0         A 2026-07-01  NORMAL            2
1         A 2026-07-02   ERROR            5
2         A 2026-07-03   ERROR            6
3         B 2026-07-01  NORMAL            1
4         B 2026-07-02  NORMAL            2
5         B 2026-07-03   ERROR            4
6         C 2026-07-01   ERROR            7
7         C 2026-07-02  NORMAL            3
8         E 2026-07-01   ERROR            9
9         E 2026-07-02   ERROR            8

df_device:
  device_id site device_type model
0         A  R34         VIS  FS11
1         B  R34         RVR  LT31
2         C  R35         VIS  FS11
3         D  R35         RVR  LT31


## 要求：

* **只保留两张表都能匹配上的设备日志。**

## 输出字段：

- `device_id`
- `stat_date`
- `status`
- `alarm_count`
- `site`
- `device_type`
- `model`

In [6]:
# ======================
# SQL轨道
# ======================

query = """
SELECT
    lo.device_id,
    lo.stat_date,
    lo.status,
    lo.alarm_count,
    dv.site,
    dv.device_type,
    dv.model
FROM df_log AS lo
INNER JOIN df_device AS dv
ON lo.device_id = dv.device_id
ORDER BY lo.device_id, lo.stat_date;
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,stat_date,status,alarm_count,site,device_type,model
0,A,2026-07-01,NORMAL,2,R34,VIS,FS11
1,A,2026-07-02,ERROR,5,R34,VIS,FS11
2,A,2026-07-03,ERROR,6,R34,VIS,FS11
3,B,2026-07-01,NORMAL,1,R34,RVR,LT31
4,B,2026-07-02,NORMAL,2,R34,RVR,LT31
5,B,2026-07-03,ERROR,4,R34,RVR,LT31
6,C,2026-07-01,ERROR,7,R35,VIS,FS11
7,C,2026-07-02,NORMAL,3,R35,VIS,FS11


In [7]:
# ======================
# PANDAS轨道
# ======================

df_pd = (
    df_log
    .merge(
        df_device,
        how= 'inner',
        on = 'device_id'
    )
    [
        [
            'device_id',
            'stat_date',
            'status',
            'alarm_count',
            'site',
            'device_type',
            'model'
        ]
    ]
    .sort_values(by=['device_id', 'stat_date'])
    .reset_index(drop=True)
)
df_pd

,device_id,stat_date,status,alarm_count,site,device_type,model
0,A,2026-07-01,NORMAL,2,R34,VIS,FS11
1,A,2026-07-02,ERROR,5,R34,VIS,FS11
2,A,2026-07-03,ERROR,6,R34,VIS,FS11
3,B,2026-07-01,NORMAL,1,R34,RVR,LT31
4,B,2026-07-02,NORMAL,2,R34,RVR,LT31
5,B,2026-07-03,ERROR,4,R34,RVR,LT31
6,C,2026-07-01,ERROR,7,R35,VIS,FS11
7,C,2026-07-02,NORMAL,3,R35,VIS,FS11
